# TASK 5 — Logistic Regression for Flight Delay Prediction

**Input:** Task 4 fitted preprocessing artifacts (sparse/dense matrices, labels, fitted preprocessor and feature names). **Evaluation:** use January–August training, September–October validation, November–December held-out test. No resplitting or refitting preprocessing here.

## 1. Load Task 4 artifacts

Upload or unzip your `task4_outputs` folder into `/content/task4_outputs/`. This avoids the optional 512 MB CSV export and uses the exact numerical imputation, scaling and one-hot encoding produced in Task 4.

In [ ]:

import zipfile
from pathlib import Path

zip_path = Path("/content/task4_outputs.zip")

assert zip_path.exists(), "Upload task4_outputs.zip first!"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content")

print("ZIP extracted successfully!")

# Check extracted files
folder = Path("/content/task4_outputs")

assert folder.is_dir(), "task4_outputs folder not found!"

print("\nExtracted files:")
for file in sorted(folder.iterdir()):
    print(file.name)


ZIP extracted successfully!

Extracted files:
X_test_cat_dl.npy
X_test_num_dl.npy
X_test_processed.npz
X_train_cat_dl.npy
X_train_num_dl.npy
X_train_processed.npz
X_val_cat_dl.npy
X_val_num_dl.npy
X_val_processed.npz
dl_preprocessing.joblib
preprocessor.joblib
processed_feature_names.csv
test_metadata.csv
train_metadata.csv
val_metadata.csv
y_test.npy
y_train.npy
y_val.npy


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import sparse
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report)

OUT = Path('/content/task4_outputs')
assert OUT.is_dir(), f'Missing {OUT}. Upload or unzip your Task 4 outputs folder first.'

def load_processed(name):
    npz = OUT / f'X_{name}_processed.npz'
    npy = OUT / f'X_{name}_processed.npy'
    if npz.exists():
        return sparse.load_npz(npz).tocsr()
    if npy.exists():
        return np.load(npy, mmap_mode='r')
    raise FileNotFoundError(f'Missing processed X_{name} in {OUT}')

X_train = load_processed('train')
X_val = load_processed('val')
X_test = load_processed('test')
y_train = np.load(OUT / 'y_train.npy').astype(int)
y_val = np.load(OUT / 'y_val.npy').astype(int)
y_test = np.load(OUT / 'y_test.npy').astype(int)
feature_names = pd.read_csv(OUT / 'processed_feature_names.csv')['feature'].tolist()
preprocessor = joblib.load(OUT / 'preprocessor.joblib')

for name, X, y, expected in [('train', X_train, y_train, 217727),
                              ('val', X_val, y_val, 55628),
                              ('test', X_test, y_test, 53991)]:
    assert X.shape[0] == len(y) == expected, (name, X.shape, len(y))
    assert X.shape[1] == len(feature_names), (name, X.shape[1], len(feature_names))
    assert set(np.unique(y)).issubset({0, 1})
    if sparse.issparse(X):
        assert np.isfinite(X.data).all(), f'{name} contains NaN or Inf'
    else:
        assert np.isfinite(X).all(), f'{name} contains NaN or Inf'
    print(f'{name}: X={X.shape}, positives={y.mean():.2%}')
assert len(feature_names) == len(set(feature_names)), 'Duplicate feature names'
print('Task 4 artifacts verified — no resplitting or preprocessing refit.')

train: X=(217727, 356), positives=25.61%
val: X=(55628, 356), positives=15.05%
test: X=(53991, 356), positives=24.99%
Task 4 artifacts verified — no resplitting or preprocessing refit.


## 2. Verify chronological separation and feature names

The metadata files are saved by Task 4. Verify month membership independently of matrix shapes. Note that `row_id` is only a tracking column, not a predictor.

In [ ]:
expected_months = {'train': set(range(1, 9)), 'val': {9, 10}, 'test': {11, 12}}
for name in ('train', 'val', 'test'):
    path = OUT / f'{name}_metadata.csv'
    md = pd.read_csv(path, parse_dates=['date'])
    assert set(md['split'].unique()) == {name}, f'Unexpected split in {path}'
    assert set(md['date'].dt.month.unique()) == expected_months[name], f'Wrong months in {name}'
    assert len(md) == len({'train': y_train, 'val': y_val, 'test': y_test}[name])
    assert np.array_equal(md['delayed'].to_numpy(dtype=int),
                           {'train': y_train, 'val': y_val, 'test': y_test}[name]), f'{name} label ordering mismatch'
    print(f'{name}: {md.date.min().date()} to {md.date.max().date()} — {len(md):,} records')
for forbidden in ['num__arr_delay', 'num__dep_delay', 'num__arr_time', 'num__dep_time',
                  'num__air_time', 'num__row_id', 'num__delayed']:
    assert forbidden not in feature_names, f'Leakage column: {forbidden}'
print('Chronology and common leakage checks passed.')

train: 2013-01-01 to 2013-08-31 — 217,727 records
val: 2013-09-01 to 2013-10-31 — 55,628 records
test: 2013-11-01 to 2013-12-31 — 53,991 records
Chronology and common leakage checks passed.


## 3. Train Logistic Regression

Fit only on the training matrix. `class_weight="balanced"` addresses class imbalance; evaluate its precision/recall trade-off on validation. The input matrices have already been transformed in Task 4.

In [ ]:
model = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42,
                           solver='lbfgs')
model.fit(X_train, y_train)
print('Model fitted. Iterations:', model.n_iter_.tolist())
if np.any(model.n_iter_ >= model.max_iter):
    print('Warning: solver may not have converged; examine convergence and adjust max_iter.')

Model fitted. Iterations: [113]


## 4. Validation metrics (threshold = 0.5)

ROC AUC and PR AUC use probabilities; precision, recall and F1 use the fixed 0.5 threshold. Compare PR AUC with the validation positive-class prevalence.

In [ ]:
def evaluate_split(name, X, y, threshold=0.5):
    proba = model.predict_proba(X)[:, 1]
    pred = (proba >= threshold).astype(int)
    metrics = {
        'split': name, 'threshold': threshold, 'n': int(len(y)),
        'positive_rate': float(y.mean()),
        'accuracy': accuracy_score(y, pred),
        'precision': precision_score(y, pred, zero_division=0),
        'recall': recall_score(y, pred, zero_division=0),
        'f1': f1_score(y, pred, zero_division=0),
        'roc_auc': roc_auc_score(y, proba),
        'pr_auc': average_precision_score(y, proba),
    }
    print(f'\n{name.upper()}')
    print(pd.Series(metrics).to_string())
    print('Confusion matrix:\n', confusion_matrix(y, pred))
    print(classification_report(y, pred, zero_division=0))
    return metrics, proba

val_metrics, val_prob = evaluate_split('val', X_val, y_val)


VAL
split                 val
threshold             0.5
n                   55628
positive_rate    0.150518
accuracy           0.6178
precision        0.218036
recall           0.595127
f1               0.319147
roc_auc          0.650957
pr_auc           0.235002
Confusion matrix:
 [[29384 17871]
 [ 3390  4983]]
              precision    recall  f1-score   support

           0       0.90      0.62      0.73     47255
           1       0.22      0.60      0.32      8373

    accuracy                           0.62     55628
   macro avg       0.56      0.61      0.53     55628
weighted avg       0.79      0.62      0.67     55628



In [ ]:

import numpy as np
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(
    y_val, val_prob
)

f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("Best validation threshold:", round(best_threshold, 4))
print("Best validation F1:", round(f1_scores[best_idx], 4))


Best validation threshold: 0.4721
Best validation F1: 0.3201


In [ ]:

from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    confusion_matrix
)

# Get test probabilities from the already-trained model
test_prob = model.predict_proba(X_test)[:, 1]

# Apply threshold selected on validation data
test_pred_tuned = (
    test_prob >= best_threshold
).astype(int)

print("Tuned Test Accuracy:",
      accuracy_score(y_test, test_pred_tuned))

print("Tuned Test Precision:",
      precision_score(y_test, test_pred_tuned, zero_division=0))

print("Tuned Test Recall:",
      recall_score(y_test, test_pred_tuned, zero_division=0))

print("Tuned Test F1:",
      f1_score(y_test, test_pred_tuned, zero_division=0))

print("Tuned Confusion Matrix:")
print(confusion_matrix(y_test, test_pred_tuned))


Tuned Test Accuracy: 0.6044525939508436
Tuned Test Precision: 0.32479279921575616
Tuned Test Recall: 0.54024607174622
Tuned Test F1: 0.4056882061557299
Tuned Confusion Matrix:
[[25346 15153]
 [ 6203  7289]]


## 5. Final held-out test evaluation

Do not use test results to tune preprocessing, hyperparameters or thresholds. This notebook reports the predeclared 0.5 threshold for both validation and test.

In [ ]:
test_metrics, test_prob = evaluate_split('test', X_test, y_test)
results = pd.DataFrame([val_metrics, test_metrics])
display(results)


TEST
split                test
threshold             0.5
n                   53991
positive_rate    0.249894
accuracy         0.632883
precision        0.334397
recall           0.473614
f1               0.392013
roc_auc          0.618679
pr_auc            0.33873
Confusion matrix:
 [[27780 12719]
 [ 7102  6390]]
              precision    recall  f1-score   support

           0       0.80      0.69      0.74     40499
           1       0.33      0.47      0.39     13492

    accuracy                           0.63     53991
   macro avg       0.57      0.58      0.56     53991
weighted avg       0.68      0.63      0.65     53991



,split,threshold,n,positive_rate,accuracy,precision,recall,f1,roc_auc,pr_auc
0,val,0.5,55628,0.150518,0.617800,0.218036,0.595127,0.319147,0.650957,0.235002
1,test,0.5,53991,0.249894,0.632883,0.334397,0.473614,0.392013,0.618679,0.338730


In [ ]:

import numpy as np
from sklearn.metrics import (
    precision_recall_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Step 1: Select threshold using validation data ONLY
precision, recall, thresholds = precision_recall_curve(
    y_val, val_prob
)

f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("Best validation threshold:", round(best_threshold, 4))
print("Best validation F1:", round(f1_scores[best_idx], 4))

# Step 2: Apply validation-selected threshold to test probabilities
test_pred_tuned = (test_prob >= best_threshold).astype(int)

print("\nTUNED TEST RESULTS")
print("Accuracy:", round(accuracy_score(y_test, test_pred_tuned), 4))
print("Precision:", round(precision_score(y_test, test_pred_tuned, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, test_pred_tuned, zero_division=0), 4))
print("F1:", round(f1_score(y_test, test_pred_tuned, zero_division=0), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred_tuned))
display(results)

Best validation threshold: 0.4721
Best validation F1: 0.3201

TUNED TEST RESULTS
Accuracy: 0.6045
Precision: 0.3248
Recall: 0.5402
F1: 0.4057

Confusion Matrix:
[[25346 15153]
 [ 6203  7289]]


,split,threshold,n,positive_rate,accuracy,precision,recall,f1,roc_auc,pr_auc
0,val,0.5,55628,0.150518,0.617800,0.218036,0.595127,0.319147,0.650957,0.235002
1,test,0.5,53991,0.249894,0.632883,0.334397,0.473614,0.392013,0.618679,0.338730


In [ ]:

from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score,
    average_precision_score
)

tuned_row = {
    "split": "test_tuned",
    "threshold": float(best_threshold),
    "n": len(y_test),
    "positive_rate": float(np.mean(y_test)),
    "accuracy": accuracy_score(y_test, test_pred_tuned),
    "precision": precision_score(y_test, test_pred_tuned, zero_division=0),
    "recall": recall_score(y_test, test_pred_tuned, zero_division=0),
    "f1": f1_score(y_test, test_pred_tuned, zero_division=0),
    "roc_auc": roc_auc_score(y_test, test_prob),
    "pr_auc": average_precision_score(y_test, test_prob),
}

results = pd.concat(
    [results[results["split"] != "test_tuned"],
     pd.DataFrame([tuned_row])],
    ignore_index=True
)


## 6. Save a reproducible model bundle

The Logistic Regression model expects Task 4-transformed features. Save the fitted Task 4 preprocessor **with** the model to support predictions on raw 21-feature inputs.

In [ ]:

MODEL_OUT = Path('/content/task5_outputs')
MODEL_OUT.mkdir(parents=True, exist_ok=True)

bundle = {
    'model': model,
    'preprocessor': preprocessor,
    'feature_names': feature_names,
    'threshold': float(best_threshold),
    'input_note': 'preprocessor.transform expects original 21 Task 3 model features',
}

joblib.dump(
    bundle,
    MODEL_OUT / 'logistic_regression_bundle.joblib'
)

results.to_csv(
    MODEL_OUT / 'logistic_regression_metrics.csv',
    index=False
)

print('Saved:', list(MODEL_OUT.iterdir()))


Saved: [PosixPath('/content/task5_outputs/logistic_regression_bundle.joblib'), PosixPath('/content/task5_outputs/logistic_regression_metrics.csv')]


## 7. Download outputs (small)

Download only model + metrics. Do not export a huge dense CSV of one-hot features.

In [ ]:
from google.colab import files
import shutil
archive = shutil.make_archive('/content/Task5_Logistic_Regression', 'zip',
                              root_dir='/content', base_dir='task5_outputs')
print('Archive size (MB):', round(Path(archive).stat().st_size / (1024**2), 2))
files.download(archive)

Archive size (MB): 0.01


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Notes for the report
- The dataset is from 2013 and the target is arrival delay greater than 15 minutes for completed flights.
- Historical features in Task 3 use a conservative lag, which remains an assumption about prediction-time availability.
- Missing-value imputation, scaling and one-hot encoding were **fitted only on the training partition in Task 4**.
- Task 5 uses chronological partitions, no random re-splitting.
- Hold out the test partition for final evaluation; if optimizing a decision threshold later, choose it on validation data only.
- Training and evaluation metrics appear only after executing this notebook in Colab with Task 4 artifacts.